# SDCP: Self-Distilled Contrastive Priors for RAG

**Novel Contribution:** LLMs self-distill question-specific contrastive priors — P_pos (confident initial answer) and P_neg (anticipated misconception) — to guide contrastive retrieval **without any test label access**.

**Compared to prior work:**
| Method | Contrastive Signal Source | Label-free? |
|--------|--------------------------|-------------|
| FLARE (2023) | None (only decides when to retrieve) | ✓ |
| Self-RAG (2024) | Fine-tuned reflection tokens | ✓ but needs fine-tuning |
| Li et al. (2025) | KB labels (correct/incorrect pairs) | ✗ needs KB labels |
| **SDCP (ours)** | **Self-distilled from model parametrics** | **✓ fully label-free** |

**Datasets:** TruthfulQA · MMLU  
**Model:** Mistral-7B-Instruct-v0.2 (fp16, local)

In [1]:
# ── Cell 1: Environment check ────────────────────────────────────────────────
import sys, torch, importlib

required = ['transformers','sentence_transformers','faiss','datasets',
            'rouge_score','mauve','sklearn','tqdm','bitsandbytes','accelerate']
missing = [p for p in required if not importlib.util.find_spec(p)]
if missing: print('MISSING:', missing)
else: print('All packages OK')

print(f'Python: {sys.version[:6]}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')
if torch.cuda.is_available():
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')

All packages OK
Python: 3.10.2
PyTorch: 2.5.1+cu121
CUDA: True | GPU: Quadro RTX 8000
VRAM: 47.5 GB


In [2]:
# ── Cell 2: Config ───────────────────────────────────────────────────────────
import os, sys

BASE_DIR     = '/home/user/RAG_Best_Practices/RAG_best_practices-main'
MODELS_DIR   = '/home/user/RAG_Best_Practices/models'
DATASETS_DIR = '/home/user/RAG_Best_Practices/datasets'
OUTPUT_DIR   = '/home/user/RAG_Best_Practices/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.chdir(BASE_DIR)
sys.path.insert(0, BASE_DIR)

# DEV MODE  : N_TQA=100, MMLU_PER_SUBJECT=5  → ~35min
# PAPER MODE: N_TQA=817, MMLU_PER_SUBJECT=32 → ~4h
N_TRUTHFULQA     = 100
MMLU_PER_SUBJECT = 5
RANDOM_SEED      = 42
QUANT            = None   # None=fp16, '4bit' for low VRAM

# SDCP hyperparameters
# cert_threshold=0.65: avg_cert=0.72 → ~45% questions use full contrastive path
# (was 0.25 → only 12% uncertain, contrastive path barely activated)
SDCP_PARAMS = {
    'cert_threshold': 0.65,   # below → uncertain → full contrastive context
    'alpha': 0.45,            # query similarity weight
    'beta':  0.35,            # P_pos similarity weight
    'gamma': 0.20,            # P_neg repulsion weight
    'top_k_retrieve': 15,     # candidates to retrieve from KB
    'top_k_context': 4,       # top sentences to use as context
    'max_gen_tokens': 25,
    'max_pos_tokens': 20,
    'max_neg_tokens': 20,
}

print('Config OK')
print(f'TruthfulQA: {N_TRUTHFULQA}Q | MMLU: {MMLU_PER_SUBJECT}×57={MMLU_PER_SUBJECT*57}Q | seed={RANDOM_SEED}')
print(f'cert_threshold={SDCP_PARAMS["cert_threshold"]} → ~45% questions will use full contrastive path')

Config OK
TruthfulQA: 100Q | MMLU: 5×57=285Q | seed=42
cert_threshold=0.65 → ~45% questions will use full contrastive path


In [3]:
# ── Cell 3: Load datasets (zero-overlap train/test split) ────────────────────
import gc, numpy as np, pandas as pd
from datasets import load_from_disk

CHOICE_LABELS = ['A', 'B', 'C', 'D']

# ── TruthfulQA
print('Loading TruthfulQA...')
tqa_raw = load_from_disk(f'{DATASETS_DIR}/truthfulqa').to_pandas()
tqa_all = tqa_raw[['question','best_answer','correct_answers','incorrect_answers']].copy()
tqa_all['correct_answers']   = tqa_all['correct_answers'].apply(lambda x: x.tolist() if isinstance(x, np.ndarray) else [x])
tqa_all['incorrect_answers'] = tqa_all['incorrect_answers'].apply(lambda x: x.tolist() if isinstance(x, np.ndarray) else [x])
tqa_all['best_answer']       = tqa_all['best_answer'].apply(lambda x: [x] if x else [])
tqa_all = tqa_all[(tqa_all['correct_answers'].apply(len) > 1) &
                   (tqa_all['incorrect_answers'].apply(len) > 1)].reset_index(drop=True)

tqa_test_idx = tqa_all.sample(n=N_TRUTHFULQA, random_state=RANDOM_SEED).index
tqa     = tqa_all.loc[tqa_test_idx].reset_index(drop=True)    # test set
tqa_icl = tqa_all.drop(tqa_test_idx).reset_index(drop=True)   # ICL KB (no overlap)
del tqa_raw; gc.collect()
print(f'  test={len(tqa)}Q | KB={len(tqa_icl)}Q (zero overlap)')

# ── MMLU (stratified)
print('Loading MMLU...')
mmlu_raw = load_from_disk(f'{DATASETS_DIR}/mmlu').to_pandas()

def mmlu_to_unified(row):
    choices  = list(row['choices'])
    ans_idx  = int(row['answer'])
    correct  = choices[ans_idx]
    incorrect = [choices[i] for i in range(len(choices)) if i != ans_idx]
    formatted_q = (row['question'] + '\n' +
                   '\n'.join(f'{CHOICE_LABELS[i]}) {choices[i]}' for i in range(len(choices))))
    return pd.Series({
        'question': formatted_q, 'question_plain': row['question'],
        'subject': row['subject'], 'best_answer': [correct],
        'correct_answers': [correct], 'incorrect_answers': incorrect,
        'answer_idx': ans_idx, 'choices': choices
    })

mmlu_test_parts, mmlu_icl_parts = [], []
for subject, group in mmlu_raw.groupby('subject'):
    group = group.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    n_test = min(MMLU_PER_SUBJECT, len(group))
    mmlu_test_parts.append(group.iloc[:n_test])
    if len(group) > n_test: mmlu_icl_parts.append(group.iloc[n_test:])

mmlu     = pd.concat(mmlu_test_parts).reset_index(drop=True).apply(mmlu_to_unified, axis=1)
mmlu_icl = pd.concat(mmlu_icl_parts).reset_index(drop=True).apply(mmlu_to_unified, axis=1)
del mmlu_raw; gc.collect()
print(f'  test={len(mmlu)}Q ({mmlu["subject"].nunique()} subjects) | KB={len(mmlu_icl)}Q (zero overlap)')
print('Datasets ready ✓')

Loading TruthfulQA...
  test=100Q | KB=615Q (zero overlap)
Loading MMLU...
  test=285Q (57 subjects) | KB=1539Q (zero overlap)
Datasets ready ✓


In [4]:
# ── Cell 4: Load model ───────────────────────────────────────────────────────
import gc, torch, psutil
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer

def show_mem():
    ram = psutil.virtual_memory()
    vram = torch.cuda.memory_allocated()/1024**3 if torch.cuda.is_available() else 0
    print(f'  RAM {ram.used/1024**3:.1f}/{ram.total/1024**3:.1f}GB | VRAM {vram:.1f}GB')

gc.collect(); torch.cuda.empty_cache(); show_mem()

print('Loading Mistral-7B...')
if QUANT == '4bit':
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    llm = AutoModelForCausalLM.from_pretrained(f'{MODELS_DIR}/mistral-7b',
                                                quantization_config=bnb, device_map='auto')
else:
    llm = AutoModelForCausalLM.from_pretrained(f'{MODELS_DIR}/mistral-7b',
                                                torch_dtype=torch.float16, device_map='auto')
tokenizer = AutoTokenizer.from_pretrained(f'{MODELS_DIR}/mistral-7b', padding_side='left')
tokenizer.pad_token = tokenizer.eos_token
show_mem()

print('Loading MiniLM...')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
embed_model = SentenceTransformer(f'{MODELS_DIR}/minilm').to(DEVICE)
show_mem()
print('Models ready!')

  RAM 10.2/47.0GB | VRAM 0.0GB
Loading Mistral-7B...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

  RAM 10.9/47.0GB | VRAM 13.5GB
Loading MiniLM...
  RAM 10.9/47.0GB | VRAM 13.6GB
Models ready!


In [5]:
# ── Cell 5: Core utilities ───────────────────────────────────────────────────
import faiss, numpy as np, pandas as pd, torch, gc
from rouge_score import rouge_scorer as rs_module
from sklearn.metrics.pairwise import cosine_similarity
import mauve as mauve_lib
from tqdm import tqdm

INST_S = '[INST]'
INST_E = '[/INST]'
SYS    = 'You are a truthful expert question-answering bot and should correctly and concisely answer the following question'

def generate(prompts, max_new_tokens=25, do_sample=False, temperature=1.0, top_p=0.9, num_beams=1):
    enc = tokenizer(prompts, return_tensors='pt', padding=True,
                    truncation=True, max_length=2048).to(DEVICE)
    input_len = enc['input_ids'].shape[1]
    kw = dict(input_ids=enc['input_ids'], attention_mask=enc['attention_mask'],
              max_new_tokens=max_new_tokens, pad_token_id=tokenizer.eos_token_id)
    if do_sample: kw.update(do_sample=True, temperature=temperature, top_p=top_p)
    else: kw['num_beams'] = num_beams
    with torch.no_grad():
        out = llm.generate(**kw)
    return [tokenizer.decode(r[input_len:], skip_special_tokens=True).strip() or 'I have no comment'
            for r in out]

def get_token_probs(prompt, max_new_tokens=25):
    enc = tokenizer(prompt, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = llm.generate(**enc, max_new_tokens=max_new_tokens,
                           return_dict_in_generate=True, output_scores=True,
                           pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(out.sequences[0][enc['input_ids'].shape[1]:],
                            skip_special_tokens=True).strip()
    cert = 0.0
    if out.scores:
        probs = torch.softmax(out.scores[0][0], dim=-1)
        top2  = torch.topk(probs, 2).values
        cert  = (top2[0] - top2[1]).item()
    return text, cert

def build_index(dataset):
    embs = embed_model.encode(dataset['question'].tolist(), show_progress_bar=True, batch_size=64)
    embs = np.array(embs, dtype=np.float32)
    faiss.normalize_L2(embs)
    idx = faiss.IndexFlatIP(embs.shape[1])
    idx.add(embs)
    return idx

def retrieve_from_kb(query, faiss_idx, kb_dataset, k=1):
    q_emb = np.array(embed_model.encode([query], show_progress_bar=False), dtype=np.float32)
    faiss.normalize_L2(q_emb)
    _, idxs = faiss_idx.search(q_emb, k)
    return [kb_dataset.iloc[i] for i in idxs[0] if i < len(kb_dataset)]

def clean_response(resp):
    for stop in ['\nQuestion:','\nQ:','\n---','\nIncorrect','\nCorrect',
                 '\nVERIFIED','\nExample','\n\n','\nMy initial']:
        if stop in resp: resp = resp[:resp.index(stop)]
    return resp.strip().strip('"').strip("'") or 'I have no comment'

def compute_metrics(generated, references):
    scorer = rs_module.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)
    r1s, r2s, rls, ecss = [], [], [], []
    for gen, refs in zip(generated, references):
        best_r1 = best_r2 = best_rl = 0
        for ref in refs:
            if not ref: continue
            s = scorer.score(ref, gen)
            best_r1 = max(best_r1, s['rouge1'].fmeasure)
            best_r2 = max(best_r2, s['rouge2'].fmeasure)
            best_rl = max(best_rl, s['rougeL'].fmeasure)
        r1s.append(best_r1*100); r2s.append(best_r2*100); rls.append(best_rl*100)
        try:
            embs = embed_model.encode([refs[0], gen])
            ecss.append(float(cosine_similarity([embs[0]], [embs[1]])[0][0])*100)
        except: ecss.append(0.0)
    return np.array(r1s), np.array(r2s), np.array(rls), np.array(ecss)

def compute_mauve(generated, references):
    try:
        refs  = [r[0] if isinstance(r, list) else r for r in references]
        valid = [(g, r) for g, r in zip(generated, refs) if g and r]
        if len(valid) < 10: return 0.0
        gens_v, refs_v = zip(*valid)
        result = mauve_lib.compute_mauve(p_text=list(refs_v), q_text=list(gens_v),
                                          device_id=0, max_text_length=256, verbose=False,
                                          featurize_model_name='gpt2')
        return result.mauve * 100
    except Exception as e:
        print(f'  MAUVE error: {e}'); return 0.0

def compute_accuracy(generated, dataset):
    correct = 0
    for gen, (_, row) in zip(generated, dataset.iterrows()):
        correct_text = row['best_answer'][0] if isinstance(row['best_answer'], list) else row['best_answer']
        ans_idx = int(row.get('answer_idx', -1))
        if ans_idx >= 0:
            label = CHOICE_LABELS[ans_idx]
            if label in gen[:5] or correct_text.lower() in gen.lower(): correct += 1
        else:
            if correct_text.lower() in gen.lower(): correct += 1
    return correct / len(generated) * 100

print('Utilities ready ✓')

Utilities ready ✓


In [6]:
# ── Cell 6: SDCP Core — Self-Distilled Contrastive Priors ───────────────────
#
# KEY IDEA: Instead of relying on KB labels or test ground truth,
# we ask the model to generate its own:
#   P_pos = what the model currently believes is correct (parametric knowledge)
#   P_neg = what common misconception exists for this question type
#
# These self-distilled priors then drive contrastive retrieval:
#   score(d) = α·sim(d,q) + β·sim(d,P_pos) - γ·sim(d,P_neg)
#
# This is FULLY label-free at test time — no access to correct/incorrect answers.

def generate_sdcp_priors(query, dataset_name, params):
    """
    Self-distill P_pos and P_neg for the given question.
    P_pos: model's confident initial answer (parametric knowledge prior)
    P_neg: model's anticipated misconception (negative prior)
    cert:  token-level certainty of P_pos (top1_prob - top2_prob)
    ZERO test label access — purely self-knowledge.
    """
    # P_pos: model's best initial guess + certainty score
    pos_prompt = (f'{INST_S}{SYS}\nQuestion: {query}\nAnswer concisely:{INST_E}')
    p_pos, cert = get_token_probs(pos_prompt, max_new_tokens=params['max_pos_tokens'])
    p_pos = clean_response(p_pos)

    # P_neg: anticipated misconception — question_plain avoids biasing toward listed choices
    q_plain = query.split('\n')[0] if '\n' in query else query
    if dataset_name == 'MMLU':
        neg_prompt = (f'{INST_S}What is a plausible but INCORRECT answer that students '
                      f'commonly give for this type of question?\n'
                      f'Question: {q_plain}\n'
                      f'Common wrong answer (very short):{INST_E}')
    else:
        neg_prompt = (f'{INST_S}What is a common misconception or false belief '
                      f'that people hold about this topic?\n'
                      f'Question: {q_plain}\n'
                      f'Common wrong belief (very short):{INST_E}')

    p_neg_raw = generate([neg_prompt], max_new_tokens=params['max_neg_tokens'], num_beams=1)[0]
    p_neg = clean_response(p_neg_raw)

    return p_pos, p_neg, cert


def run_sdcp(test_data, kb_data, dataset_name, params=None):
    """
    SDCP: Self-Distilled Contrastive Priors for Retrieval-Augmented Generation

    Step 1 — Self-distill P_pos and P_neg (no labels)
    Step 2 — Prior-guided contrastive retrieval from KB
    Step 3 — Certainty-adaptive prompt construction:
              cert < θ → uncertain → full contrastive context + KB anchor + both priors
              cert ≥ θ → confident → light retrieval + P_pos refinement + P_neg avoidance
    Step 4 — Final generation
    """
    if params is None: params = SDCP_PARAMS
    print(f'\n=== SDCP on {dataset_name} | test={len(test_data)}Q  KB={len(kb_data)}Q ===')
    print(f'    α={params["alpha"]} β={params["beta"]} γ={params["gamma"]} '
          f'cert_θ={params["cert_threshold"]}')

    print('Building KB index...')
    faiss_idx = build_index(kb_data)

    generated, references, prior_log = [], [], []

    for idx, row in tqdm(test_data.iterrows(), total=len(test_data), desc='SDCP'):
        query       = row['question']
        best_answer = row['best_answer'] if isinstance(row['best_answer'], list) else [row['best_answer']]

        # ── Step 1: Self-distill priors (zero label access)
        p_pos, p_neg, cert = generate_sdcp_priors(query, dataset_name, params)

        # ── Step 2: Prior-guided contrastive retrieval
        retrieved = retrieve_from_kb(query, faiss_idx, kb_data, k=params['top_k_retrieve'])

        prompt = None
        if retrieved and p_pos:
            sentences = []
            for doc_row in retrieved:
                sentences.append(doc_row['question'])
                ba = doc_row['best_answer']
                ba_text = ba[0] if isinstance(ba, list) and ba else str(ba)
                if ba_text and len(ba_text) > 3: sentences.append(ba_text)
            sentences = [s for s in sentences if s and len(s.strip()) > 5]

            if sentences:
                # Contrastive reranking with self-distilled priors
                s_embs   = embed_model.encode(sentences, show_progress_bar=False)
                q_emb    = embed_model.encode([query],   show_progress_bar=False)
                pos_emb  = embed_model.encode([p_pos],   show_progress_bar=False)
                neg_emb  = embed_model.encode([p_neg],   show_progress_bar=False) if p_neg else None

                q_sims   = cosine_similarity(s_embs, q_emb).flatten()
                pos_sims = cosine_similarity(s_embs, pos_emb).flatten()
                neg_sims = (cosine_similarity(s_embs, neg_emb).flatten()
                            if neg_emb is not None else np.zeros(len(sentences)))

                sdcp_scores = (params['alpha'] * q_sims +
                               params['beta']  * pos_sims -
                               params['gamma'] * neg_sims)

                top_idx = np.argsort(-sdcp_scores)[:params['top_k_context']]
                context = ' '.join([sentences[i] for i in top_idx])

                # KB example for contrastive grounding (from KB, not test)
                kb_ex = retrieved[0]
                kb_ba = kb_ex['best_answer']
                kb_correct   = kb_ba[0] if isinstance(kb_ba, list) and kb_ba else str(kb_ba)
                kb_ia = kb_ex['incorrect_answers']
                kb_incorrect = kb_ia[0] if isinstance(kb_ia, list) and kb_ia else ''
                kb_q = (kb_ex['question_plain']
                        if 'question_plain' in kb_ex.index else kb_ex['question'])

                # ── Step 3: Certainty-adaptive prompt
                if cert < params['cert_threshold']:
                    # UNCERTAIN → full contrastive context + KB anchor + both priors
                    prompt = (f'{INST_S}{SYS}\n'
                              f'Retrieved context: {context}\n'
                              f'Example — Q: {kb_q}\n'
                              f'  Correct: {kb_correct}\n'
                              f'  Incorrect: {kb_incorrect}\n'
                              f'My initial thought: {p_pos}\n'
                              f'Common mistake to avoid: {p_neg}\n'
                              f'Question: {query}\n'
                              f'Verified answer:{INST_E}')
                else:
                    # CONFIDENT → light retrieval + prior refinement + P_neg avoidance
                    prompt = (f'{INST_S}{SYS}\n'
                              f'Context: {context}\n'
                              f'Initial answer: {p_pos}\n'
                              f'Common mistake to avoid: {p_neg}\n'
                              f'Question: {query}\n'
                              f'Refined answer:{INST_E}')

        if prompt is None:
            prompt = f'{INST_S}{SYS}\nQuestion: {query}\nAnswer:{INST_E}'

        # ── Step 4: Final generation
        resp  = generate([prompt], max_new_tokens=params['max_gen_tokens'], num_beams=2)
        final = clean_response(resp[0])

        generated.append(final)
        references.append(best_answer)
        prior_log.append({'p_pos': p_pos, 'p_neg': p_neg, 'cert': cert,
                          'mode': 'uncertain' if cert < params['cert_threshold'] else 'confident'})

        if idx % 30 == 0: gc.collect(); torch.cuda.empty_cache()

    # ── Metrics
    r1, r2, rl, ecs = compute_metrics(generated, references)
    mauve_score     = compute_mauve(generated, references)

    # ── Prior quality analysis
    scorer_p = rs_module.RougeScorer(['rouge1'], use_stemmer=True)
    pos_r1_list = []
    for log, refs in zip(prior_log, references):
        ref = refs[0] if isinstance(refs, list) else refs
        if log['p_pos'] and ref:
            pos_r1_list.append(scorer_p.score(ref, log['p_pos'])['rouge1'].fmeasure * 100)

    uncertain_n  = sum(1 for l in prior_log if l['mode'] == 'uncertain')
    pos_quality  = float(np.mean(pos_r1_list)) if pos_r1_list else 0.0
    avg_cert     = float(np.mean([l['cert'] for l in prior_log]))

    print(f'Prior P_pos quality (R1 vs ref): {pos_quality:.1f}')
    print(f'Uncertain (cert<{params["cert_threshold"]}): '
          f'{uncertain_n}/{len(test_data)} ({uncertain_n/len(test_data)*100:.0f}%) | '
          f'avg cert={avg_cert:.3f}')

    results = {
        'method': f'SDCP-{dataset_name}', 'dataset': dataset_name,
        'R1': float(r1.mean()), 'R2': float(r2.mean()),
        'RL': float(rl.mean()), 'ECS': float(ecs.mean()),
        'MAUVE': mauve_score,
        'generated': generated, 'references': references,
        'prior_log': prior_log,
        'pos_quality': pos_quality,
        'uncertain_pct': uncertain_n / len(test_data) * 100,
        'avg_cert': avg_cert
    }

    if 'answer_idx' in test_data.columns:
        acc = compute_accuracy(generated, test_data)
        results['Accuracy'] = acc
        print(f'R1={r1.mean():.2f} R2={r2.mean():.2f} RL={rl.mean():.2f} '
              f'ECS={ecs.mean():.2f} MAUVE={mauve_score:.2f} Acc={acc:.1f}%')
    else:
        print(f'R1={r1.mean():.2f} R2={r2.mean():.2f} RL={rl.mean():.2f} '
              f'ECS={ecs.mean():.2f} MAUVE={mauve_score:.2f}')

    return results

print('SDCP ready ✓')

SDCP ready ✓


In [7]:
# ── Cell 7: Run SDCP on both datasets ────────────────────────────────────────
import time, json

sdcp_results = {}

print('=' * 60)
print('SDCP — TruthfulQA')
print('=' * 60)
t0 = time.time()
sdcp_results['SDCP_TQA'] = run_sdcp(tqa, tqa_icl, 'TruthfulQA')
print(f'  Time: {(time.time()-t0)/60:.1f} min')

print('\n' + '=' * 60)
print('SDCP — MMLU')
print('=' * 60)
t0 = time.time()
sdcp_results['SDCP_MMLU'] = run_sdcp(mmlu, mmlu_icl, 'MMLU')
print(f'  Time: {(time.time()-t0)/60:.1f} min')

print('\nSDCP complete!')

SDCP — TruthfulQA

=== SDCP on TruthfulQA | test=100Q  KB=615Q ===
    α=0.45 β=0.35 γ=0.2 cert_θ=0.65
Building KB index...


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

SDCP: 100%|██████████| 100/100 [04:07<00:00,  2.48s/it]


Featurizing p:   0%|          | 0/100 [00:00<?, ?it/s]

Featurizing q:   0%|          | 0/100 [00:00<?, ?it/s]

...

SDCP: 100%|██████████| 285/285 [12:52<00:00,  2.71s/it]


Featurizing p:   0%|          | 0/285 [00:00<?, ?it/s]

Featurizing q:   0%|          | 0/285 [00:00<?, ?it/s]

WARNING clustering 570 points to 28 centroids: please provide at least 1092 training points


Prior P_pos quality (R1 vs ref): 34.1
Uncertain (cert<0.65): 89/285 (31%) | avg cert=0.737
R1=33.77 R2=24.69 RL=32.39 ECS=50.25 MAUVE=57.68 Acc=47.0%
  Time: 13.0 min

SDCP complete!


In [8]:
# ── Cell 8: Results + Comparison Table ───────────────────────────────────────
from datetime import datetime

PAPER_BASELINES = {
    'TruthfulQA': {
        'Base (paper)':      {'R1':26.81,'R2':13.26,'RL':23.86,'ECS':56.44,'MAUVE':72.92},
        'ICL1D+ (paper)':    {'R1':30.62,'R2':17.45,'RL':27.79,'ECS':58.96,'MAUVE':73.86},
        '80Doc80S (paper)':  {'R1':28.85,'R2':15.01,'RL':25.51,'ECS':57.25,'MAUVE':74.15},
        # Clean results from our CFR/CAFD-LC/FACTS-RV runs (zero leakage)
        'CFR (ours)':        {'R1':38.81,'R2':23.95,'RL':35.74,'ECS':69.35,'MAUVE':12.55},
        'CAFD-LC (ours)':    {'R1':40.66,'R2':25.22,'RL':37.47,'ECS':68.00,'MAUVE':17.97},
        'FACTS-RV (ours)':   {'R1':37.62,'R2':22.22,'RL':34.43,'ECS':68.14,'MAUVE':11.84},
    },
    'MMLU': {
        'Base (paper)':      {'R1':10.42,'R2':1.90,'RL':8.91, 'ECS':29.41,'MAUVE':40.51,'Acc':None},
        'ICL1D+ (paper)':    {'R1':26.01,'R2':17.46,'RL':24.90,'ECS':47.04,'MAUVE':37.24,'Acc':None},
        '120Doc120S (paper)':{'R1':10.87,'R2':2.09,'RL':9.23, 'ECS':30.22,'MAUVE':38.88,'Acc':None},
        'CFR (ours)':        {'R1':38.56,'R2':28.52,'RL':37.44,'ECS':54.01,'MAUVE':62.73,'Acc':50.88},
        'CAFD-LC (ours)':    {'R1':38.32,'R2':28.27,'RL':37.43,'ECS':54.61,'MAUVE':46.09,'Acc':47.37},
        'FACTS-RV (ours)':   {'R1':26.86,'R2':16.08,'RL':25.18,'ECS':47.94,'MAUVE':46.43,'Acc':45.61},
    }
}

def print_comparison(dataset_name, sdcp_key):
    r = sdcp_results[sdcp_key]
    print(f'\n{"="*80}')
    print(f'RESULTS: {dataset_name}')
    print(f'{"="*80}')
    is_mmlu = dataset_name == 'MMLU'
    hdr = f'{"Method":<24} {"R1":>6} {"R2":>6} {"RL":>6} {"ECS":>6} {"MAUVE":>7}'
    if is_mmlu: hdr += f' {"Acc":>7}'
    print(hdr)
    print('-' * 80)
    for name, vals in PAPER_BASELINES[dataset_name].items():
        line = (f'{name:<24} {vals["R1"]:>6.2f} {vals["R2"]:>6.2f} '
                f'{vals["RL"]:>6.2f} {vals["ECS"]:>6.2f} {vals["MAUVE"]:>7.2f}')
        if is_mmlu:
            acc_str = f'{vals["Acc"]:>6.1f}%' if vals.get('Acc') else '      —'
            line += acc_str
        print(line)
    print('─' * 80)
    line = (f'{"SDCP (ours)":<24} {r["R1"]:>6.2f} {r["R2"]:>6.2f} '
            f'{r["RL"]:>6.2f} {r["ECS"]:>6.2f} {r["MAUVE"]:>7.2f}')
    if is_mmlu and 'Accuracy' in r:
        line += f' {r["Accuracy"]:>6.1f}%'
    print(line + ' ◄ NEW')

print_comparison('TruthfulQA', 'SDCP_TQA')
print_comparison('MMLU', 'SDCP_MMLU')

# ── Prior analysis summary
print('\n── Prior Analysis ──────────────────────────────────')
for key in ['SDCP_TQA', 'SDCP_MMLU']:
    r = sdcp_results[key]
    print(f'{r["method"]}:')
    print(f'  P_pos quality (R1 before refinement): {r["pos_quality"]:.1f}')
    print(f'  Uncertain questions: {r["uncertain_pct"]:.0f}% (cert < {SDCP_PARAMS["cert_threshold"]})')
    print(f'  Avg certainty: {r["avg_cert"]:.3f}')


RESULTS: TruthfulQA
Method                       R1     R2     RL    ECS   MAUVE
--------------------------------------------------------------------------------
Base (paper)              26.81  13.26  23.86  56.44   72.92
ICL1D+ (paper)            30.62  17.45  27.79  58.96   73.86
80Doc80S (paper)          28.85  15.01  25.51  57.25   74.15
CFR (ours)                38.81  23.95  35.74  69.35   12.55
CAFD-LC (ours)            40.66  25.22  37.47  68.00   17.97
FACTS-RV (ours)           37.62  22.22  34.43  68.14   11.84
────────────────────────────────────────────────────────────────────────────────
SDCP (ours)               34.08  18.65  30.93  65.80   12.26 ◄ NEW

RESULTS: MMLU
Method                       R1     R2     RL    ECS   MAUVE     Acc
--------------------------------------------------------------------------------
Base (paper)              10.42   1.90   8.91  29.41   40.51      —
ICL1D+ (paper)            26.01  17.46  24.90  47.04   37.24      —
120Doc120S (paper)    

In [9]:
# ── Cell 9: Save results ─────────────────────────────────────────────────────
from datetime import datetime
import json

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
summary = {}
for key, r in sdcp_results.items():
    summary[key] = {
        'method': r['method'], 'dataset': r['dataset'],
        'R1': round(r['R1'], 3), 'R2': round(r['R2'], 3),
        'RL': round(r['RL'], 3), 'ECS': round(r['ECS'], 3),
        'MAUVE': round(r['MAUVE'], 3),
        'Accuracy': round(r.get('Accuracy', 0), 2),
        'pos_quality': round(r['pos_quality'], 2),
        'uncertain_pct': round(r['uncertain_pct'], 1),
        'avg_cert': round(r['avg_cert'], 4)
    }

out_path = f'{OUTPUT_DIR}/sdcp_summary_{ts}.json'
with open(out_path, 'w') as f:
    json.dump(summary, f, indent=2)

for key, r in sdcp_results.items():
    df = pd.DataFrame({
        'generated': r['generated'],
        'reference': [x[0] if isinstance(x, list) else x for x in r['references']],
        'p_pos':  [l['p_pos']  for l in r['prior_log']],
        'p_neg':  [l['p_neg']  for l in r['prior_log']],
        'cert':   [l['cert']   for l in r['prior_log']],
        'mode':   [l['mode']   for l in r['prior_log']]
    })
    df.to_csv(f'{OUTPUT_DIR}/{key}_{ts}.csv', index=False, quoting=1)

print(f'Saved to {OUTPUT_DIR}/')
print(json.dumps(summary, indent=2))

Saved to /home/user/RAG_Best_Practices/outputs/
{
  "SDCP_TQA": {
    "method": "SDCP-TruthfulQA",
    "dataset": "TruthfulQA",
    "R1": 34.077,
    "R2": 18.647,
    "RL": 30.927,
    "ECS": 65.803,
    "MAUVE": 12.261,
    "Accuracy": 0,
    "pos_quality": 33.24,
    "uncertain_pct": 33.0,
    "avg_cert": 0.7243
  },
  "SDCP_MMLU": {
    "method": "SDCP-MMLU",
    "dataset": "MMLU",
    "R1": 33.77,
    "R2": 24.692,
    "RL": 32.387,
    "ECS": 50.251,
    "MAUVE": 57.68,
    "Accuracy": 47.02,
    "pos_quality": 34.07,
    "uncertain_pct": 31.2,
    "avg_cert": 0.7371
  }
}
